In [0]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

import math

In [0]:
# path = '/kaggle/input/datasets/sehaj1104/student-mental-health-and-burnout-dataset/student_mental_health_burnout.csv'
path = '../data/raw/student_mental_health_burnout_relabeled.csv'

# Load data & Exploring

In [0]:
df = pd.read_csv(path)
df.head()

In [0]:
df.info()
df = df.drop('student_id', axis=1)

In [0]:
df.describe(include='all')

In [0]:
num_col = df.select_dtypes(exclude='object')
cat_col = df.select_dtypes(include='object')

In [0]:
n_cols = len(num_col.columns)
no_row_plot = math.ceil(n_cols / 3)

fig, axes = plt.subplots(no_row_plot, 3, figsize=(14, no_row_plot * 4))
axes_flat = axes.flatten()

for i, col_name in enumerate(num_col.columns):
    sns.histplot(x=num_col[col_name], ax=axes_flat[i], kde=True)
    axes_flat[i].set_title(f'Countplot of {col_name}')

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')

plt.tight_layout()
plt.show();

In [0]:
n_cols = len(cat_col.columns)
no_row_plot = math.ceil(n_cols / 3)

fig, axes = plt.subplots(no_row_plot, 3, figsize=(14, no_row_plot * 4))
axes_flat = axes.flatten()

for i, col_name in enumerate(cat_col.columns):
    sns.countplot(x=cat_col[col_name], ax=axes_flat[i])
    axes_flat[i].set_title(f'Countplot of {col_name}')

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')

plt.tight_layout()
plt.show();

In [0]:
df = df.dropna()
df.isnull().sum()

In [0]:
# Onehot Encoding

nomial_col = ['course', 'gender']

df = pd.get_dummies(df, columns=nomial_col, drop_first=True)
df.head()

In [0]:
bool_col = df.select_dtypes(include='bool').columns
df[bool_col] = df[bool_col].astype('int')

df.info()

In [0]:
# Label Encoding

ordinal_col = ['year', 'stress_level', 'sleep_quality', 'internet_quality', 'burnout_level']

df[ordinal_col] = df[ordinal_col].replace({    
    '1st': 1, '2nd': 2, '3rd': 3, '4th': 4,
    'Low': 0, 'Medium': 1, 'High': 2,
    'Poor': 0, 'Average': 1, 'Good': 2
})

df.info()

In [0]:
plt.figure(figsize=(8, 14))

sns.heatmap(
    df.corr(method='pearson', numeric_only=True)[['burnout_level']].sort_values(by='burnout_level', ascending=False),
    annot=True,
    cmap='coolwarm',
    vmin=-1, vmax=1
)

## Kiểm tra đa cộng tuyến trên tập dữ liệu đã được xử lý (chỉ số VIF)

In [0]:
df = df.drop('gender_Other', axis=1)
X = df.drop('burnout_level', axis=1)
X.head()

In [0]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = X.dropna()
y = df['burnout_level'].copy()

pd.DataFrame(
    {
        'Feature': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }
).sort_values(by='VIF', ascending=False)

In [0]:
# Calculate grid dimensions
n_cols = 3
n_rows = math.ceil(len(df.columns) / n_cols)

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes_flat = axes.flatten()

# Create boxplot for each numerical feature
for i, feature in enumerate(df.columns):
    sns.boxplot(data=df, x='burnout_level', y=feature, ax=axes_flat[i], palette='Set2')
    axes_flat[i].set_title(f'{feature} vs Burnout Level', fontsize=12, fontweight='bold')
    axes_flat[i].set_xlabel('Burnout Level (0=Low, 1=Medium, 2=High)')
    axes_flat[i].set_ylabel(feature)
    axes_flat[i].grid(axis='y', alpha=0.3)

# Hide unused subplots
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')

plt.tight_layout()
plt.show()

# Training model (Random Forest, LightGBM, XGBoost)

In [0]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

import lightgbm as lgb
import xgboost as xgb


def evaluate_models_cv(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    models = {
        "RandomForest": RandomForestClassifier(random_state=42),
        "LightGBM": lgb.LGBMClassifier(random_state=42),
        "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    }

    results = []

    for model_name, model in models.items():
        f1s, recs, precs = [], [], []
        train_times, pred_times = [], []

        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Train
            start_train = time.time()
            model.fit(X_train, y_train)
            train_times.append(time.time() - start_train)

            # Predict
            start_pred = time.time()
            preds = model.predict(X_test)
            pred_times.append(time.time() - start_pred)

            # Metrics
            precs.append(precision_score(y_test, preds, average='weighted'))
            recs.append(recall_score(y_test, preds, average='weighted'))
            f1s.append(f1_score(y_test, preds, average='weighted'))

        results.append({
            "Model": model_name,
            "Precision": np.mean(precs),
            "Recall": np.mean(recs),
            "F1": np.mean(f1s),
            "Train time": np.mean(train_times),
            "Predict time": np.mean(pred_times)
        })

    return pd.DataFrame(results), models

In [0]:
df_eval, models = evaluate_models_cv(X, y)

In [0]:
df_eval

In [0]:
N_RUNS = 50
SUBSET_SIZE = 0.5  

records = []

for i in range(N_RUNS):
    X_sub, _, y_sub, _ = train_test_split(
        X_test, y_test,
        test_size=1 - SUBSET_SIZE,
        random_state=i,
        stratify=y_test
    )

    rec_lgb  = recall_score(y_sub, models['LightGBM'].predict(X_sub), average='weighted')
    rec_xgb = recall_score(y_sub, models['XGBoost'].predict(X_sub), average='weighted')

    records.append({
        'run'         : i + 1,
        'LightGBM' : round(rec_lgb,  6),
        'XGB': round(rec_xgb, 6),
    })


recall_df = pd.DataFrame(records)
recall_df.head()

### Áp dụng kiểm định để so sánh 2 mô hình

In [0]:
recall_df['XGB - LGB'] = recall_df['XGB_Accuracy'] - recall_df['LightGBM']

sns.histplot(recall_df['XGB - LGB'], bins=50)
plt.show();

Đặt ra giả thuyết: "Accuracy của mô hình XGBoost là lớn hơn LightGBM"

Gọi hiệu số giữa Accuracy của XGBoost và LightGBM là d, cặp giả thuyết thống kê là:
H0:μd=0H_0: \mu_d = 0
H1:μd>0H_1: \mu_d > 0
Chọn mức ý nghĩa là $\alpha = 0.05$. Thực hiện kiểm định giả thuyết trên

In [0]:
from scipy import stats

t_stat, p_val = stats.ttest_1samp(recall_df['XGB - LGB'], 0)
alpha = 0.05
print(f'T statistic: {t_stat}\np-value: {p_val}')

print(f'Bác bỏ H0 vì p-value < alpha ({p_val} < {alpha})' if p_val < alpha else f'Không bác bỏ H0 vì p-value >= alpha ({p_val} > {alpha})')

# Hyperparameter Fine-tuning

In [0]:
import optuna
from xgboost.callback import EarlyStopping

from xgboost.callback import EarlyStopping

def objective(trial):
    param = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "random_state": 42,

        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),

        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train
    )

    model = xgb.XGBClassifier(**param)

    model.fit(
        X_train_sub, y_train_sub,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)

    score = recall_score(y_val, preds, average='weighted') 

    return score

In [0]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print("Best Parameters:", study.best_params)

In [0]:
optuna.visualization.plot_optimization_history(study).show()

In [0]:
optuna.visualization.plot_param_importances(study).show()